# Unified Silver Cryptocurrency Prices
This notebook combines cleaned historical OHLC data and streaming price ticks into one canonical Silver table for analytics and dashboard development.


## Available Silver tables

- `team_crypto_silver.ohlc_prices` — historical hourly OHLC data from Kraken 
(taken from 22.08.2026 07:00 to 21.09.13:00).
- `team_crypto_silver.prices` — cleaned streaming price ticks from CoinGecko and Event Hub.
- `team_crypto_silver.unified_prices` — the unified analytics-ready dataset created by this notebook.

The unified table preserves the data source and uses `record_type` to distinguish historical candles from streaming ticks.

### 1. Configuration

In [0]:
# creating parameters
dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("silver_schema", "team_crypto_silver")

catalog = dbutils.widgets.get("catalog")
silver_schema = dbutils.widgets.get("silver_schema")

batch_table = f"{catalog}.{silver_schema}.ohlc_prices"
streaming_table = f"{catalog}.{silver_schema}.prices"
target_table = f"{catalog}.{silver_schema}.unified_prices"

print("Batch table:", batch_table)
print("Streaming table:", streaming_table)
print("Target table:", target_table)

## 2. Source Validation

Verify that both source tables are available and inspect their schemas and latest records before transformation.

In [0]:
print("BATCH TABLE SCHEMA")
spark.table(batch_table).printSchema()

print("STREAMING TABLE SCHEMA")
spark.table(streaming_table).printSchema()

In [0]:
from pyspark.sql import functions as F

print("BATCH DATA")
display(
    spark.table(batch_table)
    .orderBy(F.col("event_time").desc())
    .limit(5)
)

print("STREAMING DATA")
display(
    spark.table(streaming_table)
    .orderBy(F.col("event_time").desc())
    .limit(5)
)

## 3. Prepare Historical OHLC Data

Select the columns required for analytics and use the closing price (`close`) as the common `price_usd` value. 

Historical rows retain their OHLC and volume values and are marked as `ohlc_batch`.

In [0]:
batch_df = (
    spark.table(batch_table)
    .select(
        F.col("symbol"),
        F.col("event_time"),
        F.col("close").alias("price_usd"),
        F.col("open"),
        F.col("high"),
        F.col("low"),
        F.col("close"),
        F.col("volume"),
        F.lit("ohlc_batch").alias("record_type"),
        F.col("source")
    )
)


## 4. Prepare Streaming Price Ticks

Normalize cryptocurrency names to the common symbols `BTC`, `ETH`, and `SOL`.

Streaming events contain a current price but do not contain complete OHLC candles. Therefore, the OHLC and volume columns remain `NULL` for these rows.

In [0]:
streaming_df = (
    spark.table(streaming_table)
    .withColumn(
        "symbol",
        F.when(F.lower(F.col("symbol")) == "bitcoin", "BTC")
         .when(F.lower(F.col("symbol")) == "ethereum", "ETH")
         .when(F.lower(F.col("symbol")) == "solana", "SOL")
         .otherwise(F.upper(F.col("symbol")))
    )
    .select(
        F.col("symbol"),
        F.col("event_time"),
        F.col("price_usd"),
        F.lit(None).cast("double").alias("open"),
        F.lit(None).cast("double").alias("high"),
        F.lit(None).cast("double").alias("low"),
        F.lit(None).cast("double").alias("close"),
        F.lit(None).cast("double").alias("volume"),
        F.lit("streaming_tick").alias("record_type"),
        F.col("source")
    )
)


## 5. Build the Unified Dataset


In [0]:
unified_df = (
    batch_df
    .unionByName(streaming_df)
    .dropDuplicates([
        "symbol",
        "event_time",
        "record_type"
    ])
)

## 6. Validate the Unified Dataset

In [0]:
print("Batch rows:", batch_df.count())
print("Streaming rows:", streaming_df.count())
print("Unified rows:", unified_df.count())

display(
    unified_df
    .orderBy(F.col("event_time").desc())
    .limit(20)
)

In [0]:
display(
    unified_df
    .groupBy("record_type", "symbol")
    .count()
    .orderBy("record_type", "symbol")
)

## 7. Write to the Unified Silver Table

Create or incrementally update `team_crypto_silver.unified_prices`. The Delta `MERGE` makes the operation idempotent: rerunning the notebook inserts only records that are not already present.

In [0]:
from delta.tables import DeltaTable

if spark.catalog.tableExists(target_table):

    rows_before = spark.table(target_table).count()

    target_delta = DeltaTable.forName(spark, target_table)

    (target_delta.alias("target")
        .merge(
            unified_df.alias("source"),
            """
            target.symbol = source.symbol
            AND target.event_time = source.event_time
            AND target.record_type = source.record_type
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    action = "MERGE completed"

else:

    rows_before = 0

    (
        unified_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )

    action = "Unified Silver table created"

rows_after = spark.table(target_table).count()

print(action)
print("Target table:", target_table)
print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("New rows inserted:", rows_after - rows_before)

## 8. Final Data Quality Check

In [0]:
display(
    spark.table(target_table)
    .groupBy("record_type", "symbol")
    .count()
    .orderBy("record_type", "symbol")
)